In [ ]:
#This is pyenv file for making AI classification.

In [1]:
import os
import pandas as pd
from IPython.display import display, Markdown
from tqdm.auto import tqdm # makes pretty progress bars
tqdm.pandas()  
from dotenv import load_dotenv
load_dotenv()
import time

In [17]:
from tqdm.auto import tqdm
tqdm.pandas()

In [2]:
data=pd.read_csv('data2000.csv')

In [3]:
data_dir=''
SIZE_OF_SAMPLE_FOR_HANDCODING = 50
text_column_name = "DESC_DEFECT"
data.dropna(subset=[text_column_name], inplace=True)

In [4]:
data_sample = data.sample(n=SIZE_OF_SAMPLE_FOR_HANDCODING, random_state=613)

data_sample = data_sample.copy() # avoid annoying warnings!
data_sample.head()

,RECORD_ID,CAMPNO,MAKETXT,MODELTXT,YEARTXT,MFGCAMPNO,COMPNAME,MFGNAME,BGMAN,ENDMAN,...,DESC_DEFECT,CONEQUENCE_DEFECT,CORRECTIVE_ACTION,NOTES,RCL_CMPT_ID,MFR_COMP_NAME,MFR_COMP_DESC,MFR_COMP_PTNO,PARK_IT,PARK_OUTSIDE
3409,66938,07V074000,SKYLINE,CELEBRITY,2000,NaN,EQUIPMENT:RECREATIONAL VEHICLE/TRAILER,SKYLINE CORPORATION,19970401.0,20030831.0,...,"CERTAIN TRUCK CAMPERS, FIFTH WHEEL AND TRAVEL ...",RELEASE OF COOLANT UNDER CERTAIN CONDITIONS CO...,SKYLINE WILL BE WORKING WITH DOMETIC IN ORDER ...,CUSTOMERS MAY CONTACT THE NATIONAL HIGHWAY TRA...,23262000221524000000330,NaN,NaN,NaN,No,No
3166,62057,06V183000,VOLKSWAGEN,BEETLE,2000,N4,EXTERIOR LIGHTING:BRAKE LIGHTS:SWITCH,"VOLKSWAGEN OF AMERICA, INC",19980101.0,20011130.0,...,ON CERTAIN PASSENGER VEHICLES EQUIPPED WITH AU...,"THE BRAKE LIGHTS COULD BECOME INOPERATIVE, OR ...",DEALERS WILL INSTALL THE NEWLY DESIGNED BRAKE ...,VW RECALL NO. N4.CUSTOMERS MAY ALSO CONTACT TH...,22751000084731000000228,NaN,NaN,NaN,No,No
2959,52901,04V311000,SPARTAN,SUMMIT,2000,04011,"SERVICE BRAKES, HYDRAULIC:FOUNDATION COMPONENTS","SPARTAN CHASSIS, INC.",19990225.0,20020521.0,...,CERTAIN MY 1999-2002 SPARTAN SUMMIT MOTOR HOME...,OVERHEATED BRAKE COMPONENTS MAY NOT ALLOW THE ...,SPARTAN WILL REPLACE THE BRAKE CALIPERS AND PA...,SPARTAN RECALL NO. 04011.CUSTOMERS CAN ALSO CO...,21240000093419000000071,NaN,NaN,NaN,No,No
3182,63026,07V417000,REXHALL,AMERICAN CLIPPER,2000,NaN,EQUIPMENT:RECREATIONAL VEHICLE/TRAILER,REXHALL INDUSTRIES INC,19980601.0,20030630.0,...,ON CERTAIN MOTOR HOMES EQUIPPED WITH A TWO-DOO...,RELEASE OF COOLANT UNDER CERTAIN CONDITIONS CO...,REXHALL WILL BE WORKING WITH DOMETIC IN ORDER ...,CUSTOMERS MAY CONTACT THE NATIONAL HIGHWAY TRA...,24812000110584000000330,NaN,NaN,NaN,No,No
3601,74980,08V025000,NEWMAR,SCOTTSDALE,2000,05S28,VEHICLE SPEED CONTROL,NEWMAR CORPORATION,NaN,NaN,...,ON CERTAIN CLASS A MOTOR HOMES CONSTRUCTED ON ...,OVERHEATING COULD RESULT IN SMOKE OR BURN AND ...,NEWMAR IS WORKING WITH FORD IN CONDUCTING THIS...,NEWMAR RECALL NO. 05S28.CUSTOMERS MAY ALSO CON...,25854000110442000000294,NaN,NaN,NaN,No,No


In [5]:
from openrouter import OpenRouter
openrouter_client = OpenRouter(api_key=os.environ.get("OPENROUTER_API_KEY_LEDE"))

In [6]:
prompt_base = """

This is a data car recall data Summary from NHTSA.
Please read this and classify what caused the recall
, choosing from just these options: {categories}. Return only the topic name,
nothing else.
if it is caused by software issue, please select 'software' first.
Software includes driving assist system(ADAS).

Summary is {Summary}

"""

from enum import Enum, IntEnum
class RecallOptions(str, Enum):
    Software = "software"
    Safety_Equipment = "safety_equipment"
    Powertrain = "powertrain"
    Chasies = "chasies"
    Electronics="electronics"
    other_reason="others"
    
data_sample_prompt_column = data_sample.apply(lambda row: prompt_base.format(
    Summary=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in RecallOptions])
), axis="columns")
full_data_prompt_column = data.apply(lambda row: prompt_base.format(
    Summary=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in RecallOptions])
), axis="columns")

In [7]:
from strip_tags import strip_tags
import tiktoken
def count_tokens(model, text):
  if "claude" in model and os.environ.get("ANTHROPIC_API_KEY_LEDE"):
    return anthropic_client.messages.count_tokens(model=model, messages=[{"content": text, "role": "user"}])
  elif "gpt" in model:
    encoding = tiktoken.encoding_for_model('gpt-4o') # most modern models use the same tokenizer
    tokens = encoding.encode(text)
    return len(tokens)
  else:
    return count_tokens("gpt-4o", text) # for estimation purposes, pretend mistral is chatgpt...

INPUT_TOKEN_COSTS = {
    "gpt-5-mini": 0.40 / 1_000_000,  # https://openai.com/api/pricing/
    "gpt-5.4-mini": 0.75 / 1_000_000,
    "gpt-5.4": 2.5 / 1_000_000,
    "gpt-5.5": 5 / 1_000_000,
    "gpt-5.4-nano": 0.2 / 1_000_000,
    "mistral-medium-latest": 1.5 / 1_000_000, # https://mistral.ai/pricing#api-pricing
    "mistral-large-latest":  0.5 / 1_000_000,  # No, I don't know why medium is more expensive than large.
    "claude-haiku-4.5": 1 / 1_000_000,
    "claude-sonnet-4.6": 3 / 1_000_000
}
# ignores outputs, since the model will output fairly little text

def estimate_cost(model, token_count):
    return token_count * INPUT_TOKEN_COSTS[model]

In [8]:
DEFAULT_MODEL_TO_USE = 'gpt-5.4-nano'
display(Markdown("Pick a model to use: \n" + "\n - ".join(list(INPUT_TOKEN_COSTS.keys()))))
MODEL_TO_USE = None

while MODEL_TO_USE not in INPUT_TOKEN_COSTS.keys():
  MODEL_TO_USE = input(f"type a model name (press enter for default: {DEFAULT_MODEL_TO_USE}): ").strip()
  if MODEL_TO_USE == "":
    MODEL_TO_USE = DEFAULT_MODEL_TO_USE

Pick a model to use: 
gpt-5-mini
 - gpt-5.4-mini
 - gpt-5.4
 - gpt-5.5
 - gpt-5.4-nano
 - mistral-medium-latest
 - mistral-large-latest
 - claude-haiku-4.5
 - claude-sonnet-4.6

type a model name (press enter for default: gpt-5.4-nano):  gpt-5.4-mini


In [9]:
count_tokens_for_our_model = lambda text: count_tokens(MODEL_TO_USE, text)

token_count_sample = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(data_sample_prompt_column))
"Sample would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_sample))

'Sample would cost: $0.01'

In [10]:
## cost estimation for full dataset
token_count_full = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(full_data_prompt_column))
"Full dataset would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_full))

'Full dataset would cost: $0.51'

In [11]:
from pydantic import BaseModel


class RecallValidOptions(BaseModel):
  classification: RecallOptions

SYSTEM_PROMPT = "You are a helpful assistant"

def chatgpt_classify(prompt_including_tweet):
    # put our prompt into the blob that OpenAI expects
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt_including_tweet,
        }
    ]
    chat_completion = openai_client.responses.parse(
        input=messages,
        model=MODEL_TO_USE,
        text_format=MuskTweetValidOptions,
    )

    # get the answer out of the blob that OpenAI returns.
    resp = chat_completion.output_parsed.classification.value if chat_completion.output_parsed else None
    return resp

openrouter_model_names = {
    "gpt-5-mini": "openai/gpt-5-mini",
    "gpt-5.4-mini": "openai/gpt-5.4-mini",
    "gpt-5.4": "openai/gpt-5.4",
    "gpt-5.5": "openai/gpt-5.5",
    "gpt-5.4-nano": "openai/gpt-5.4-nano",
    "mistral-medium-latest": "mistralai/mistral-medium-3-5",
    "mistral-large-latest": "mistralai/mistral-large-2512",
    "claude-haiku-4.5": "anthropic/claude-4.5-haiku",
    "claude-sonnet-4.6": "anthropic/claude-4.6-sonnet"
}

def openrouter_classify(prompt_including_tweet):
    response = openrouter_client.chat.send(
        model=openrouter_model_names[MODEL_TO_USE],
        messages=[
            {"role": "user", "content": prompt_including_tweet}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "recall_classification",
                "strict": True,
                "schema": RecallValidOptions.model_json_schema(),
            },
        },
        max_completion_tokens=64,
        temperature=0,
    )
    return  RecallValidOptions.model_validate_json(
        response.choices[0].message.content
    ).classification.value

def classify(prompt_including_tweet):
  "A wrapper for the provider-specific classification functions"
  if USE_OPENROUTER:
    return openrouter_classify(prompt_including_tweet)
  else:
    if "mistral" in MODEL_TO_USE == "mistral-medium-latest":
      return mistral_classify(prompt_including_tweet)
    elif "claude" in MODEL_TO_USE:
      return anthropic_classify(prompt_including_tweet)
    elif "gpt" in MODEL_TO_USE:
      return chatgpt_classify(prompt_including_tweet)
    else:
      raise ValueError("Unknown model")

def classify_fix(prompt_including_tweet):
    for attempt in range(3):
        try:
            result = classify(prompt_including_tweet)

            time.sleep(0.5)

            return result
        except Exception as e:
            time.sleep(attempt*3)
    return None    

In [18]:
USE_OPENROUTER=True
# ask ChatGPT about each and every tweet IN THE SAMPLE, using the prompt we made above
data_sample["ai_guess"] = data_sample_prompt_column.progress_apply(classify_fix)

  0%|          | 0/50 [00:00<?, ?it/s]

In [13]:
with pd.option_context("display.max_colwidth", 500):
  display(
      data_sample[[text_column_name, "ai_guess"]].head(10)
  )

,DESC_DEFECT,ai_guess
3409,"CERTAIN TRUCK CAMPERS, FIFTH WHEEL AND TRAVEL TRAILERS EQUIPPED WITH A TWO-DOOR REFRIGERATOR, MANUFACTURED BY THE DOMETIC CORPORATION, MAY HAVE A DEFECT IN THE BOILER TUBE. PRESSURIZED COOLANT SOLUTION COULD BE RELEASED INTO AN AREA WHERE AN IGNITION SOURCE (GAS FLAME) IS PRESENT.",others
3166,"ON CERTAIN PASSENGER VEHICLES EQUIPPED WITH AUTOMATIC OR MANUAL TRANSMISSIONS AND CRUISE CONTROLS, A BRAKE LIGHT SWITCH ON THESE VEHICLES MAY MALFUNCTION IF IT WAS INSTALLED INCORRECTLY DURING SAFETY RECALLS 03V-265 OR 04V-096.",safety_equipment
2959,"CERTAIN MY 1999-2002 SPARTAN SUMMIT MOTOR HOMES, FRONT ENGINE SCHOOL BUSES, AND TRANSIT BUSES MANUFACTURED BETWEEN FEBRUARY 25, 1999, AND MAY 21, 2002, AND EQUIPPED WITH DANA FRONT AXLE CONTAINING BOSCH HYDRAULIC DISC BRAKES WITH ZERO OFFSET SLIDE CALIPERS (ZOPS). BOSCH HAS DECIDED THAT THE PRESENCE OF CORROSION IN THE VICINITY OF THE CALIPER SEAL LAND CAN CONTRIBUTE TO REDUCED PISTON MOBILITY IN SOME CASES AND LEAD TO PERFORMANCE ISSUES CAUSING OVERHEATING.",chasies
3182,"ON CERTAIN MOTOR HOMES EQUIPPED WITH A TWO-DOOR REFRIGERATOR MANUFACTURED BY THE DOMETIC CORPORATION, THE REFRIGERATOR MAY HAVE A DEFECT IN THE REFRIGERATOR BOILER TUBE. PRESSURIZED COOLANT SOLUTION COULD BE RELEASED INTO AN AREA WHERE AN IGNITION SOURCE (GAS FLAME) IS PRESENT.",others
3601,"ON CERTAIN CLASS A MOTOR HOMES CONSTRUCTED ON FORD F-53TY1 AND P131 MOTOR HOME CHASSIS AND EQUIPPED WITH SPEED CONTROL, THE SPEED CONTROL DEACTIVATION SWITCH MAY OVERHEAT.",electronics
3115,COUNTRY COACH HAS REPORTED THAT THE TOYO 275/70R M102Z TIRES ON CERTAIN MOTOR HOMES ARE DEFECTIVE AND COULD FAIL UNEXPECTEDLY.,others
2506,"ON CERTAIN TRAILERS EQUIPPED WITH SEALCO SPRING BRAKE CONTROL VALVE ASSEMBLIES INCORPORATED IN ARVINMERITOR AXLES, PART NOS. 110700 AND 110800, THERE IS A RECESSED AREA ON TOP OF THE VALVE IN WHICH A BRASS GUIDE PIN FITS DOWN INSIDE THE BODY OF THE VALVE AND IS HELD IN PLACE BY A STEEL SNAP RING. THE RECESSED AREA ALLOWS ACCUMULATION OF WATER, DIRT, AND/OR CHEMICALS TO BUILD UP BEHIND THE SNAP RING TO THE POINT WHERE IT WILL NO LONGER CONTAIN THE GUIDE PIN RESULTING IN A FAILURE OF THE AIR ...",safety_equipment
488,VEHICLE DESCRIPTION: HEAVY DUTY TRUCKS. THE UPPER STEERING COLUMN PINCH BOLT MAY NOT HAVE BEEN PROPERLY TORQUED.,chasies
2792,"ON CERTAIN CONVENTIONAL TRUCKS EQUIPPED WITH PRO-SLEEPER BERTHS, WATER INTRUSION CAN BE FOUND IN THE BUNK CONTROL PANEL AREA AND POOL IN THE TOPS OF THE DOME LIGHT, TV/VCR, AND OR REFRIGERATOR POWER SWITCHES.",electronics
2101,"CERTAIN MOTOR HOMES EQUIPPED WITH CERTAIN DRAG LINKS, TIE RODS AND BALL SOCKET BEARINGS, PART NO. TRW 24 DL, SUPPLIED BY HOLLAND NEWAY AND MANUFACTURED BY TRW. THESE STEERING COMPONENTS HAVE A BELOW SPECIFICATION CASE DEPTH AND/OR HARDNESS WHICH CAN LEAD TO PREMATURE WEAR OUT OF THE SOCKET IN SOME APPLICATIONS.",chasies


In [19]:
okay_to_do_the_whole_thing = input()

In [20]:
data_full_prompt_column = prompt_base + data[text_column_name]

In [21]:
data["ai_guess"] = data_full_prompt_column.progress_apply(classify_fix)
data.to_csv(data_dir + 'Recall_classified2000.csv')

  0%|          | 0/3666 [00:00<?, ?it/s]

In [16]:
import os
import requests
import json

OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY_LEDE"]

response = requests.get(
    "https://openrouter.ai/api/v1/key",
    headers={
        "Authorization": f"Bearer {OPENROUTER_API_KEY}"
    }
)

print(response.status_code)
print(json.dumps(response.json(), indent=2))

200
{
  "data": {
    "label": "sk-or-v1-fa3...4da",
    "is_management_key": false,
    "is_provisioning_key": false,
    "limit": 20,
    "limit_reset": "daily",
    "limit_remaining": 18.77052725,
    "include_byok_in_limit": false,
    "usage": 28.3285345,
    "usage_daily": 1.22947275,
    "usage_weekly": 22.67001305,
    "usage_monthly": 26.4015806,
    "byok_usage": 0,
    "byok_usage_daily": 0,
    "byok_usage_weekly": 0,
    "byok_usage_monthly": 0,
    "is_free_tier": false,
    "expires_at": null,
    "creator_user_id": "user_2uatOGNq9lBDlGjBej7lXgXiKT3",
    "rate_limit": {
      "requests": -1,
      "interval": "10s",
      "note": "This field is deprecated and safe to ignore."
    }
  }
}


In [16]:
test_results = data_sample_prompt_column.head(5).progress_apply(classify_fix)
test_results

  0%|          | 0/5 [00:00<?, ?it/s]

3409              others
3166    safety_equipment
2959             chasies
3182              others
3601         electronics
dtype: str